<a href="https://colab.research.google.com/github/B-Mohid/AI_agents-automation_assignment/blob/main/DTAA_TaxEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install dependencies
!pip install anthropic pydantic numpy

import os
import json
from pydantic import BaseModel
from anthropic import Anthropic

In [4]:
# 1. Mathematical Formula Models
class TaxArbitrageInput(BaseModel):
    nri_residence_country: str  # e.g., "USA", "UAE", "UK"
    asset_type: str            # "NRO_FD", "NRE_FD", "Mutual_Fund_LTCG"
    nominal_return_rate: float # e.g., 0.075 for 7.5%
    tenure_years: float        # e.g., 1.0
    inr_depreciation_rate: float # e.g., 0.03 (3% annual INR drift)
    foreign_marginal_tax_rate: float # e.g., 0.24 (24% US tax bracket)

def calculate_dtaa_yield(inputs: TaxArbitrageInput, dtaa_cap_rate: float, standard_tds_rate: float) -> dict:
    r = inputs.nominal_return_rate
    t_f = inputs.foreign_marginal_tax_rate

    # If UAE (0% tax regime), DTAA cap doesn't apply to foreign offset
    if inputs.nri_residence_country.upper() in ["UAE", "QATAR", "OMAN"]:
        t_f = 0.0

    # Determine Effective Indian Tax Rate
    applicable_tds = min(standard_tds_rate, dtaa_cap_rate)

    # Claimable Foreign Tax Credit (FTC) under DTAA
    ftc_claimable = min(applicable_tds, dtaa_cap_rate)

    # Net global effective tax rate
    extra_foreign_tax = max(0.0, t_f - ftc_claimable)
    total_effective_tax = applicable_tds + extra_foreign_tax

    # Currency Depreciation Multiplier (E0/Et)
    exchange_multiplier = 1.0 / ((1.0 + inputs.inr_depreciation_rate) ** inputs.tenure_years)

    # Net Return Equation
    gross_yield = (1.0 + r) ** inputs.tenure_years - 1.0
    net_yield_after_tax = gross_yield * (1.0 - total_effective_tax)
    real_effective_yield_usd = ((1.0 + net_yield_after_tax) * exchange_multiplier) - 1.0

    return {
        "gross_inr_yield_pct": round(gross_yield * 100, 2),
        "applicable_indian_tds_pct": round(applicable_tds * 100, 2),
        "ftc_claimable_pct": round(ftc_claimable * 100, 2),
        "net_global_tax_drag_pct": round(total_effective_tax * 100, 2),
        "net_effective_usd_yield_pct": round(real_effective_yield_usd * 100, 2)
    }

In [5]:
# 2. Agent Implementation using Claude
class DTAATaxEngineAgent:
    def __init__(self, api_key: str):
        self.client = Anthropic(api_key=api_key)

    def Optimize_Remittance(self, data: TaxArbitrageInput):
        prompt = f"""
        You are a Cross-Border NRI Wealth & DTAA International Tax Advisory Agent.
        Provide the bilateral treaty tax cap for the following setup:

        NRI Country of Residence: {data.nri_residence_country}
        Asset Category: {data.asset_type}

        Return STRICTLY JSON format:
        {{
            "dtaa_article": "<Article number e.g. Article 11 Interest>",
            "dtaa_cap_rate": <float e.g. 0.15 for 15% cap under US-India DTAA>,
            "standard_indian_tds": <float e.g. 0.30 for NRO 30% TDS>,
            "compliance_requirements": ["<e.g. Form 10F>", "<TRC Tax Residency Certificate>", "<Form 15CA/CB>"]
        }}
        """

        response = self.client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=800,
            temperature=0.0,
            messages=[{"role": "user", "content": prompt}]
        )

        dtaa_info = json.loads(response.content[0].text)

        math_out = calculate_dtaa_yield(
            data,
            dtaa_cap_rate=dtaa_info["dtaa_cap_rate"],
            standard_tds_rate=dtaa_info["standard_indian_tds"]
        )

        return {
            "input_summary": data.model_dump(),
            "treaty_provisions": dtaa_info,
            "mathematical_yield_analysis": math_out
        }

In [6]:
# 3. Test Run
if __name__ == "__main__":
    API_KEY = os.getenv("ANTHROPIC_API_KEY", "your-claude-api-key")

    input_case = TaxArbitrageInput(
        nri_residence_country="USA",
        asset_type="NRO_FD",
        nominal_return_rate=0.075,  # 7.5% Indian NRO FD
        tenure_years=1.0,
        inr_depreciation_rate=0.025, # 2.5% expected INR decay vs USD
        foreign_marginal_tax_rate=0.22 # 22% US Tax Bracket
    )

    agent = DTAATaxEngineAgent(api_key=API_KEY)
    # result = agent.Optimize_Remittance(input_case)
    # print(json.dumps(result, indent=2))